In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [25]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings,ChatGoogleGenerativeAI
from langchain_community.vectorstores import InMemoryVectorStore
from langchain.tools import tool
from langchain.agents import create_agent

In [55]:
loader=PyPDFLoader("../data/GenAI_Course_Content.pdf")
docs=loader.load()

In [5]:
len(docs)

9

In [58]:
splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
splitter_docs=splitter.split_documents(docs)

In [57]:
len(splitter_docs)

83

In [59]:
embeddings=GoogleGenerativeAIEmbeddings(model="gemini-embedding-2")

In [60]:
vector_store=InMemoryVectorStore.from_documents(
    documents=splitter_docs,
    embedding=embeddings
)

In [16]:
same_record=vector_store.similarity_search("What is the name of the patient?")

In [17]:
same_record[0].page_content

'Page 1: Patient Information & Overview\nPatient Name: John Doe\nAge: 45\nGender: Male\nDate of Report: 10 October 2025\nHospital: City Care Medical Center\nChief Complaint:'

In [ ]:
### agent= tools,llm,prompt

In [69]:

@tool
def retriever_tool(query:str):
    """
    This tool can help you to retrieve the relevant data of the PDF Documents, and These are the tools that you can use to retrieve the data from the PDF Documents and That pdf have details of medical reports of patients .
    """
    print("tool is called with query:", query)
    docs= vector_store.similarity_search(query=query, K=4)
    context=""
    for doc in docs:
        context =doc.page_content +"\n\n"

    return context

In [70]:
llm=ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite")

In [71]:
System_prompt="""
You are a helpful assistant that answer questions using retrieved context.
ALWAYS use the  `retriver_tool` tool for questions requiring external knowledge .
"""

In [72]:
agent= create_agent(
    model=llm,
    tools=[retriever_tool],
    system_prompt=System_prompt
)

In [77]:
query="Give me the summary of module 1 in this course ?"
response = agent.invoke({"messages": [{"role": "user", "content": query}]})

tool is called with query: what is module 1 in this course? summary of module 1?
tool is called with query: what is MODULE 1? list the topics of module 1.
tool is called with query: "MODULE 1"
tool is called with query: what is the first module mentioned in the document? show me the content of the first module.
tool is called with query: "MODULE 1" "MODULE 2" "MODULE 3"
tool is called with query: what is the title of the first module? list everything under MODULE 1.
tool is called with query: find the beginning of the course, what is module 1?
tool is called with query: "MODULE 1"
tool is called with query: What is the first module in the course? Please provide the topics included in Module 1.


In [78]:
result=response["messages"][-1].text


In [79]:
print(result)

Based on the course materials, **Module 1** is titled **"Fundamentals"**. It serves as an introduction to Generative AI and the core concepts behind Large Language Models.

Here is a summary of the topics covered in Module 1:

*   **Series Introduction:** An overview of the Generative AI series, the content roadmap, projects, and information on 1-to-1 support.
*   **What is LLM and How It Works?:** An explanation of Large Language Models, including how they are trained, an introduction to transformer architecture, an overview of popular LLMs (like GPT, Claude, Gemini, and Llama), and a discussion of their capabilities and limitations.
*   **Tokenization & Building Custom Tokenizer:** A technical breakdown of how text is processed by LLMs.
